# W4-004 — Train the Black Hole detector (Colab T4)

This notebook trains the Week 4 sentence detector using the saved W4-001 activations. It fits only on `train.npz`, uses `validation.npz` for early stopping, and does not open or evaluate `test.npz`. Checkpoints and history are written to Google Drive each epoch, so the last checkpoint can be resumed after an interrupted session.

Before running, choose **Runtime → Change runtime type → T4 GPU**. Upload the five current source files when prompted: `config.py`, `detector.py`, `loss.py`, `checkpointing.py`, and `trainer.py`. The activation files should already be in `PSSN2/artifacts/black_hole/w4-001-activations` on Drive; if train/validation files are missing, the notebook will ask you to upload those two files.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/Rami2212/PSSN-LLM-Reasoning.git'
REPO_DIR = Path('/content/PSSN-LLM-Reasoning')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

from google.colab import files
source_files = ['config.py', 'detector.py', 'loss.py', 'checkpointing.py', 'trainer.py']
print('Upload the current source files from your local src/black_hole folder:', source_files)
uploaded = files.upload()
missing = sorted(set(source_files) - set(uploaded))
if missing:
    raise ValueError(f'Please upload all five files; missing: {missing}')
package_dir = REPO_DIR / 'src' / 'black_hole'
package_dir.mkdir(parents=True, exist_ok=True)
for name in source_files:
    (package_dir / name).write_bytes(uploaded[name])
print('Updated trainer installed in', package_dir)

In [ ]:
from google.colab import drive
import torch

drive.mount('/content/drive')
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. Select Runtime → Change runtime type → T4 GPU.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name, '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
if 'T4' not in gpu_name.upper():
    raise RuntimeError(f'Expected a T4 GPU, but Colab assigned: {gpu_name}')

DRIVE_ROOT = Path('/content/drive/MyDrive/PSSN2')
DATA_DIR = DRIVE_ROOT / 'artifacts' / 'black_hole' / 'w4-001-activations'
OUTPUT_DIR = DRIVE_ROOT / 'artifacts' / 'black_hole' / 'w4-004-training'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
needed = ['train.npz', 'validation.npz']
missing = [name for name in needed if not (DATA_DIR / name).is_file()]
if missing:
    print('Upload these W4-001 files from your PC:', missing)
    uploaded_data = files.upload()
    for name in missing:
        if name not in uploaded_data:
            raise ValueError(f'Missing upload: {name}')
        (DATA_DIR / name).write_bytes(uploaded_data[name])
print('Training data:', DATA_DIR)
print('Persistent output:', OUTPUT_DIR)
for name in needed:
    print(name, (DATA_DIR / name).stat().st_size, 'bytes')

In [ ]:
import json
import sys

sys.path.insert(0, str(REPO_DIR))
from src.black_hole.trainer import train_detector

summary = train_detector(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    epochs=100,
    batch_size=32,
    patience=12,
    learning_rate=1e-3,
    weight_decay=1e-4,
    hidden_size=256,
    dropout=0.2,
    seed=42,
    use_class_weights=True,
    device='cuda',
    resume=(OUTPUT_DIR / 'last_checkpoint.pt').is_file() and not (OUTPUT_DIR / 'training_summary.json').is_file(),
)
print(json.dumps(summary, indent=2))
print('Saved files:', sorted(path.name for path in OUTPUT_DIR.iterdir()))